# Methane point-source detection: assembling real Sentinel-2 L1C + Carbon Mapper data

This notebook assembles the *data* side of Varon et al. 2024
(*Nature Communications*, [10.1038/s41467-024-47754-y](https://doi.org/10.1038/s41467-024-47754-y)) - a
methane point-source detection method built on pairs of Sentinel-2 L1C
(top-of-atmosphere) scenes, validated here against a real, live-detected leak
from [Carbon Mapper's public data platform](https://carbonmapper.org/articles/product-guide)
(`api.carbonmapper.org`) - a global, continuously updated methane/CO2
point-source monitoring system (Planet Tanager-1, NASA EMIT, and airborne
AVIRIS-NG/3 and GAO), not the older static 2020-2021 airborne-only archive
on Zenodo the paper itself used.

> **Scope, stated up front and restated at the end**: this notebook gets you
> to real validation-shaped data - a real confirmed leak, its real
> reference/detection Sentinel-2 L1C pair, and the real methane absorption
> signal between them. It does **not** reproduce the paper's synthetic
> training-data generator (a Gaussian plume dispersion model plus
> Beer-Lambert absorption injected into clean background scenes), and it
> does **not** implement or train the paper's Vision-Transformer/U-Net
> model. Both are explicitly out of scope here.

Steps:

1. Query Carbon Mapper's live plume catalog for real, recent leaks.
2. Pick one real, satellite-confirmed methane leak.
3. Find a real reference/detection Sentinel-2 L1C pair around it.
4. Download and read both scenes.
5. Tile the detection scene into fixed-size chips and find the one
   containing the leak.
6. Plot the real methane absorption signal (B12 reflectance drop) between
   the two dates.

## Why the Carbon Mapper query is plain code, not a library provider

`sentinel_analysis.providers` ships ERA5, CAMS and OpenAQ - each consumed
by the library's own fusion workflow (e.g. OpenAQ stations get
interpolated onto the analysis grid and correlated with the thermal
cube). Carbon Mapper's API is shaped the same way as those (a paginated
REST endpoint, optional API-key auth for higher limits) and is a live,
general-purpose data source, not a one-off research artifact - unlike
this notebook's first version, which used the paper's static 2020-2021
Zenodo file directly. What it still lacks is a *consumer*: nothing in
this library's downscaling/fusion pipeline does anything with detected
methane plumes yet. Without that, promoting it to a provider class would
add a live discovery API disconnected from everything else the library
does. This notebook is that missing consumer's proof of concept - reused
as real code here, and promoted to `providers/carbon_mapper.py` if a
real workflow needs it later.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Make the notebook work from the repository root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sentinel_analysis import (
    AOI,
    AnalysisGrid,
    AssetCache,
    CDSEDownloader,
    ClientConfig,
    SENTINEL2_L1C_PRODUCT_TYPE,
    Sentinel2Catalog,
    read_s2_l1c,
    select_temporal_pair,
)
from sentinel_analysis.http import http_session

OUTPUT_DIR = PROJECT_ROOT / 'output' / 'methane-detection-carbon-mapper'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

## 1. Query Carbon Mapper's live plume catalog

`GET /api/v1/catalog/plume-csv` returns a filtered CSV of detected
plumes - no API key required for public columns, an optional bearer
token (via `/api/v1/token/pair`) raises rate limits and unlocks a few
extra fields. `bbox` is passed as a repeated query parameter
(`bbox=west&bbox=south&bbox=east&bbox=north`), `datetime` as an RFC 3339
`start/end` interval. This example searches Kuwait's oil and gas fields -
one of Carbon Mapper's most consistently monitored regions - over two
real months; change `BBOX`/`DATETIME_RANGE` to search anywhere else, or a
recent rolling window, for a different real leak.

In [ ]:
import csv
import io

CARBON_MAPPER_API = 'https://api.carbonmapper.org/api/v1/catalog/plume-csv'
BBOX = (46.5, 28.5, 48.5, 30.6)  # west, south, east, north - Kuwait
DATETIME_RANGE = ('2026-07-01T00:00:00Z', '2026-08-31T23:59:59Z')

session = http_session()
response = session.get(
    CARBON_MAPPER_API,
    params=(
        [('bbox', value) for value in BBOX]
        + [
            ('datetime', '/'.join(DATETIME_RANGE)),
            ('plume_gas', 'CH4'),
            ('qualities', 'good'),
            ('sort', 'emissions_desc'),
            ('limit', 20),
        ]
    ),
    timeout=60,
)
response.raise_for_status()
plumes = list(csv.DictReader(io.StringIO(response.text)))
print(f'{len(plumes)} real, satellite-confirmed methane plumes found')
for row in plumes:
    print(f"  {row['plume_id']}: {row['datetime']} {row['place']}, {row['country']} - "
          f"{float(row['emission_auto']):.0f} +/- {float(row['emission_uncertainty_auto']):.0f} kg/h")

## 2. Pick one real, satellite-confirmed methane leak

Using the largest reported emitter in the search above - a real Tanager-1
detection over the Rawdatien oil field.

In [ ]:
plume = next(p for p in plumes if p['place'] == 'Rawdatien')
plume_lat = float(plume['plume_latitude'])
plume_lon = float(plume['plume_longitude'])
plume_date = plume['datetime'][:10]
emission_rate_kg_h = float(plume['emission_auto'])
emission_rate_uncertainty_kg_h = float(plume['emission_uncertainty_auto'])

print(f"Plume {plume['plume_id']} ({plume['instrument']}, {plume['platform']}):")
print(f'  location: {plume_lat}, {plume_lon}')
print(f'  date: {plume_date}')
print(f"  sector: {plume['ipcc_sector']}")
print(f'  reported emission rate: {emission_rate_kg_h:.0f} +/- {emission_rate_uncertainty_kg_h:.0f} kg/h')

## 3. Find a real Sentinel-2 L1C reference/detection pair

L1C, not L2A: L2A's atmospheric correction assumes a methane-free
atmosphere and would attenuate the SWIR absorption signature this relies
on. `select_temporal_pair` mirrors the paper's rule - the clearest scene
within 7 days before the leak date as the detection acquisition, and the
clearest scene 1-4 months earlier as the reference. Because this leak is
recent (not a 2020-2021 one), both acquisitions are comfortably inside
CDSE's online rolling archive - the risk flagged in `docs/roadmap.md` for
older Carbon Mapper events does not apply here.

In [ ]:
config = ClientConfig.from_env()
pad = 0.05  # roughly +/-5 km around the plume, enough for several 2.5 km chips
search_aoi = AOI(
    west=plume_lon - pad, south=plume_lat - pad,
    east=plume_lon + pad, north=plume_lat + pad,
)

catalog = Sentinel2Catalog(config, product_type=SENTINEL2_L1C_PRODUCT_TYPE)
reference_window_start = (np.datetime64(plume_date) - np.timedelta64(120, 'D')).astype(str)
candidates = catalog.search(search_aoi, reference_window_start, plume_date, cloud_cover_max=40, limit=50)
print(f'{len(candidates)} real Sentinel-2 L1C products found over this AOI/window')

pair = select_temporal_pair(
    candidates, plume_date,
    detection_lookback_days=7, reference_lookback_days=(30, 120), cloud_cover_max=25.0,
)
print(f'Reference : {pair.reference.name} (cloud_cover={pair.reference.cloud_cover})')
print(f'Detection : {pair.detection.name} (cloud_cover={pair.detection.cloud_cover})')
print(f'Gap: {pair.gap_days:.0f} days')

## 4. Download and read both scenes

A single acquisition can span more than one MGRS tile; the clearest
candidate for a given day is not guaranteed to be the tile that actually
contains the plume, so each day's same-time candidates are tried in
clear-to-cloudy order until one overlaps the requested AOI. (This
particular AOI happens to sit well inside a single tile throughout - the
fallback below is what makes that not an assumption.)

In [ ]:
downloader = CDSEDownloader(config, cache=AssetCache(OUTPUT_DIR / 'cache'))

def read_pair_scene(same_day_candidates, aoi):
    for candidate in sorted(same_day_candidates, key=lambda p: p.cloud_cover if p.cloud_cover is not None else 101.0):
        archive = downloader.download(candidate, OUTPUT_DIR / 'downloads')
        try:
            return read_s2_l1c(archive, aoi=aoi), candidate
        except ValueError:
            continue  # this tile does not cover the AOI - try the next one
    raise RuntimeError('No same-day tile covers the requested AOI')

def same_day(all_candidates, reference):
    day = reference.start_datetime[:10]
    return [p for p in all_candidates if p.start_datetime and p.start_datetime[:10] == day]

reference_ds, reference_used = read_pair_scene(same_day(candidates, pair.reference), search_aoi)
detection_ds, detection_used = read_pair_scene(same_day(candidates, pair.detection), search_aoi)
print(f'Reference tile used: {reference_used.name}')
print(f'Detection tile used: {detection_used.name}')
print(f'Reference shape: {dict(reference_ds.sizes)}')
print(f'Detection shape: {dict(detection_ds.sizes)}')

## 5. Tile into fixed-size chips and find the one over the leak

`AnalysisGrid.chips()` partitions a grid into uniform sub-grids - the
2.5x2.5 km patches the paper trains and evaluates on, as opposed to one
AOI-shaped clip. Building the grid from the detection scene's own
bounds/CRS/resolution and tiling it demonstrates how you would prepare a
full patch-based dataset from many scenes; here, only the one chip that
actually contains the real leak location matters.

In [ ]:
resolution_m = detection_ds.attrs['resolution_m']
left, right = float(detection_ds.x.min()) - resolution_m / 2, float(detection_ds.x.max()) + resolution_m / 2
bottom, top = float(detection_ds.y.min()) - resolution_m / 2, float(detection_ds.y.max()) + resolution_m / 2

scene_grid = AnalysisGrid.from_bounds((left, bottom, right, top), crs=detection_ds.attrs['crs'], resolution_m=resolution_m)
chips = scene_grid.chips(2500)  # 2.5 km, matching the paper's patch size
print(f'{len(chips)} fixed-size 2.5x2.5 km chips tile this scene')

from pyproj import Transformer
plume_x, plume_y = Transformer.from_crs('EPSG:4326', scene_grid.crs, always_xy=True).transform(plume_lon, plume_lat)
plume_chip = next(
    chip for chip in chips
    if chip.bounds[0] <= plume_x <= chip.bounds[2] and chip.bounds[1] <= plume_y <= chip.bounds[3]
)
print(f'Leak falls inside chip: {plume_chip.grid_id}')

def clip_to_chip(dataset, chip):
    return dataset.sel(
        x=slice(chip.bounds[0], chip.bounds[2]),
        y=slice(chip.bounds[3], chip.bounds[1]),
    )

reference_chip = clip_to_chip(reference_ds, plume_chip)
detection_chip = clip_to_chip(detection_ds, plume_chip)
print(f'Chip shape: {dict(detection_chip.sizes)}')

## 6. The real methane absorption signal

Methane absorbs in the SWIR, most strongly inside Sentinel-2's B12 band.
The paper's model looks for exactly this kind of localized reflectance
drop between two otherwise-similar acquisitions. This plots the real
reference and detection B12 reflectance for the chip, and their
difference, with the leak's real reported location and emission rate
annotated - not a model prediction, the actual satellite signal a
detector would be shown.

In [ ]:
b12_reference = reference_chip['B12'].isel(time=0)
b12_detection = detection_chip['B12'].isel(time=0)
b12_change = b12_detection - b12_reference

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, data, title in (
    (axes[0], b12_reference, f'Reference B12\n{reference_used.start_datetime[:10]}'),
    (axes[1], b12_detection, f'Detection B12\n{detection_used.start_datetime[:10]}'),
    (axes[2], b12_change, 'Detection - Reference'),
):
    cmap = 'RdBu' if title.startswith('Detection -') else 'gray'
    vmax = float(np.nanmax(np.abs(data))) if title.startswith('Detection -') else None
    image = ax.imshow(data.values, cmap=cmap, vmin=-vmax if vmax else None, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, fraction=0.046)

fig.suptitle(
    f"Real leak {plume['plume_id']} ({plume['place']}, {plume['country']}): "
    f'{emission_rate_kg_h:.0f} +/- {emission_rate_uncertainty_kg_h:.0f} kg/h reported',
)
fig.tight_layout()
plt.show()

## Takeaways

This notebook gets you real, paper-shaped validation data for one
confirmed leak, found through Carbon Mapper's live catalog rather than a
frozen historical export: a real Sentinel-2 L1C reference/detection
pair, the correct product level and radiometric correction (see
`docs/roadmap.md` for a real offset bug this pairing work caught and
fixed in both L1C and L2A readers), a 2.5x2.5 km chip located
automatically around the real leak coordinates, and the real B12
absorption signal - not a synthetic example.

What is still missing to fully replicate the paper (restated from the
top, deliberately, not softened by everything above):

- **The synthetic training-data generator.** The paper trains on ~1.65
  million synthetic chips: real background scenes with a Gaussian-plume
  dispersion model and Beer-Lambert absorption injected in, plus
  autocorrelated noise. Nothing here generates that - this notebook only
  assembles *real* validation-style samples, one leak at a time.
- **The deep-learning model.** A Vision Transformer encoder with a U-Net
  decoder, trained on the above. Not implemented, not in scope.
- **A real `providers/carbon_mapper.py`.** The live API queried in step 1
  is genuinely provider-shaped (like OpenAQ), but nothing in this
  library's fusion/downscaling workflow consumes detected plumes yet -
  see the note after the introduction for why that gap, not the API's
  shape, is what is keeping it out of `sentinel_analysis.providers` for
  now.